# Two ways to get a comparison model

Five FCC political-advertising invoices from
[RealKIE-FCC-Verified](https://huggingface.co/datasets/amazon-agi/RealKIE-FCC-Verified), extracted by
Claude Haiku 4.5 from OCR text, scored by stickler.

The point is the two paths to a comparison model:

- **A: you write the schema.** A hand-written Pydantic model, the usual case.
- **B: the dataset brings the schema.** The dataset ships a `json_schema` column, so the model is
  built from it and nothing is hand-written.

Both paths use the same documents and the same extraction model, so the scores are comparable.

Requires AWS credentials with Bedrock access to `us.anthropic.claude-haiku-4-5`. Ten billable calls.

In [1]:
import json
import urllib.parse
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from typing import List, Optional

import boto3
from pydantic import BaseModel, Field
from strands import Agent
from strands.models import BedrockModel
from strands_evals import Case, Experiment

from stickler.integrations.strands_evals import StructuredOutputEvaluator
from stickler.structured_object_evaluator.models.structured_model import StructuredModel

N_DOCS = 5
MAX_LINE_ITEMS = 12          # Hungarian matching is cubic in list length
MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

## The documents

In [2]:
url = (
    "https://datasets-server.huggingface.co/rows"
    f"?dataset={urllib.parse.quote('amazon-agi/RealKIE-FCC-Verified', safe='')}"
    "&config=default&split=test&offset=0&length=25"
)
with urllib.request.urlopen(url, timeout=60) as resp:
    rows = [item["row"] for item in json.load(resp)["rows"]]

rows = [r for r in rows if len(r["json_response"].get("LineItems") or []) <= MAX_LINE_ITEMS][:N_DOCS]
DATASET_SCHEMA = rows[0]["json_schema"]

print(f"{len(rows)} documents")
for r in rows:
    print(f"  {r['id'][:28]:30} {len(r['json_response'].get('LineItems') or []):>2} line items")

5 documents
  3dabc131205d3a2c74a73eb54549    4 line items
  75986ca8ed4b97420672b7782390    4 line items
  a501c2b139c99cc56e6d41a8f9df    2 line items
  64e40cb2f126bad63679f19505e8    3 line items
  08e2649c3291ee569a116260d46e    0 line items


## Path A: the schema you write

Plain Pydantic, no stickler annotations. Every nullable field is `Optional`, because on a scanned
invoice those fields are legitimately absent and "correctly returned nothing" has to score as a
success rather than a miss.

In [3]:
class FCCLineItem(BaseModel):
    LineItemDescription: Optional[str] = None
    LineItemStartDate: Optional[str] = None
    LineItemEndDate: Optional[str] = None
    LineItemDays: Optional[str] = None
    LineItemRate: Optional[float] = None


class FCCInvoice(BaseModel):
    Agency: str
    Advertiser: str
    GrossTotal: Optional[float] = None
    PaymentTerms: Optional[str] = None
    AgencyCommission: Optional[float] = None
    NetAmountDue: Optional[float] = None
    LineItems: List[FCCLineItem] = Field(default_factory=list)


print(f"hand-written: {len(FCCInvoice.model_fields)} fields")

hand-written: 7 fields


## Path B: the schema the dataset ships

One normalization is needed first. The dataset declares `"type": "float"`, which is not a JSON Schema
type, and stickler validates against the metaschema rather than guessing. Mapping it to `"number"` is
the whole fix.

The nullable fields arrive as two-branch `anyOf` (`[{"type": "float"}, {"type": "null"}]`), which is
what Pydantic emits for `Optional[T]` and what most real extraction schemas look like.

In [4]:
def normalize(node):
    """Map the dataset's non-standard `type: float` onto JSON Schema's `number`."""
    if isinstance(node, dict):
        if node.get("type") == "float":
            node = {**node, "type": "number"}
        return {k: normalize(v) for k, v in node.items()}
    if isinstance(node, list):
        return [normalize(v) for v in node]
    return node


DatasetInvoice = StructuredModel.from_json_schema(normalize(DATASET_SCHEMA))

print(f"derived from the dataset: {DatasetInvoice.__name__}")
for name, field in DatasetInvoice.model_fields.items():
    if name != "extra_fields":
        print(f"  {name:20} {field.annotation}")

derived from the dataset: DynamicModel
  Agency               <class 'str'>
  Advertiser           <class 'str'>
  GrossTotal           typing.Optional[float]
  PaymentTerms         typing.Optional[str]
  AgencyCommission     typing.Optional[float]
  NetAmountDue         typing.Optional[float]
  LineItems            typing.List[stickler.structured_object_evaluator.models.model_factory.DynamicModel]


## Extract

The same prompt and the same model for both paths. Only the `structured_output_model` differs, so the
comparison isolates the schema.

In [5]:
SESSION = boto3.Session(region_name="us-east-1")
SESSION.client("sts").get_caller_identity()      # free, fails fast on a stale token
BEDROCK = BedrockModel(model_id=MODEL, boto_session=SESSION)

PROMPT = (
    "Extract the invoice fields from this FCC political advertising invoice. Return null for any "
    "field not shown on the document. Transcribe dates exactly as printed. Include every row of the "
    "line-item table.\n\nDOCUMENT:\n{text}"
)


def extract(row, model_cls):
    agent = Agent(model=BEDROCK, system_prompt="You extract invoice data.", callback_handler=None)
    result = agent(PROMPT.format(text=row["text"][:12_000]), structured_output_model=model_cls)
    if result.structured_output is None:
        raise ValueError(f"no structured output for {row['id']}")
    return result.structured_output


def extract_all(model_cls):
    with ThreadPoolExecutor(max_workers=5) as pool:
        return dict(zip((r["id"] for r in rows), pool.map(lambda r: extract(r, model_cls), rows)))


predictions = {"A: hand-written": extract_all(FCCInvoice),
               "B: from dataset": extract_all(DatasetInvoice)}
print(f"extracted {N_DOCS} documents twice ({2 * N_DOCS} Haiku calls)")

extracted 5 documents twice (10 Haiku calls)


## Score

The ground truth is the dataset's own `json_response`, validated into whichever model the path uses.

In [6]:
ground_truth = {
    "A: hand-written": {r["id"]: FCCInvoice.model_validate(r["json_response"]) for r in rows},
    "B: from dataset": {r["id"]: DatasetInvoice.from_json(r["json_response"]) for r in rows},
}

reports, evaluators = {}, {}
for path, model_cls in (("A: hand-written", FCCInvoice), ("B: from dataset", DatasetInvoice)):
    gt, pred = ground_truth[path], predictions[path]
    cases = [Case(name=doc_id, input="", expected_output=gt[doc_id], metadata={"id": doc_id})
             for doc_id in gt]
    evaluators[path] = StructuredOutputEvaluator(model_cls)
    reports[path] = await Experiment(cases=cases, evaluators=[evaluators[path]]).run_evaluations_async(
        lambda c: pred[c.metadata["id"]])

print(f"{'document':30} {'A':>7} {'B':>7}")
print("-" * 46)
for doc_id in ground_truth["A: hand-written"]:
    a = dict(zip((c["name"] for c in reports["A: hand-written"].cases), reports["A: hand-written"].scores))
    b = dict(zip((c["name"] for c in reports["B: from dataset"].cases), reports["B: from dataset"].scores))
    print(f"{doc_id[:28]:30} {a[doc_id]:>7.3f} {b[doc_id]:>7.3f}")
print("-" * 46)
print(f"{'overall':30} {reports['A: hand-written'].overall_score:>7.3f} "
      f"{reports['B: from dataset'].overall_score:>7.3f}")

document                             A       B
----------------------------------------------
3dabc131205d3a2c74a73eb54549     0.994   0.987
75986ca8ed4b97420672b7782390     0.506   0.806
a501c2b139c99cc56e6d41a8f9df     0.943   0.939
64e40cb2f126bad63679f19505e8     0.826   0.797
08e2649c3291ee569a116260d46e     0.843   0.571
----------------------------------------------
overall                          0.822   0.820


## Which field is broken

Per-field confusion matrix for each path. **FN** is a field the extractor missed, **FA** one it
invented, **FD** one it got wrong.

Nested `LineItems.*` rows only count documents whose line-item pair scored at or above
`match_threshold`; below that, threshold gating emits no field breakdown, so those documents show up
as `fd` on `LineItems` instead.

In [7]:
for path, evaluator in evaluators.items():
    rollup = next(iter(evaluator.metrics().values()))
    print(f"\n{path}   ({rollup.document_count} documents)")
    print(f"  {'field':28} {'tp':>3} {'fn':>3} {'fa':>3} {'fd':>3}  {'f1':>5}")
    print("  " + "-" * 54)
    for field, m in sorted(rollup.field_metrics.items(), key=lambda kv: (kv[1].get("cm_f1", 1.0), kv[0])):
        print(f"  {field:28} {m.get('tp', 0):>3} {m.get('fn', 0):>3} {m.get('fa', 0):>3} "
              f"{m.get('fd', 0):>3}  {m.get('cm_f1', 0):>5.2f}")


A: hand-written   (5 documents)
  field                         tp  fn  fa  fd     f1
  ------------------------------------------------------
  LineItems                      7   0   0   6   0.70
  LineItems.LineItemDays         4   0   0   3   0.73
  Agency                         3   0   0   2   0.75
  Advertiser                     4   0   0   1   0.89
  AgencyCommission               4   0   0   1   0.89
  PaymentTerms                   4   0   1   0   0.89
  GrossTotal                     5   0   0   0   1.00
  LineItems.LineItemDescription   7   0   0   0   1.00
  LineItems.LineItemEndDate      7   0   0   0   1.00
  LineItems.LineItemRate         7   0   0   0   1.00
  LineItems.LineItemStartDate    7   0   0   0   1.00
  NetAmountDue                   5   0   0   0   1.00

B: from dataset   (5 documents)
  field                         tp  fn  fa  fd     f1
  ------------------------------------------------------
  LineItems                      5   0   3   8   0.48
  LineIte

## What the schema costs you

Both paths score the same documents against the same labels, so any gap comes from the schema rather
than the extractor.

The real difference is configuration. A hand-written model gets comparators and thresholds inferred
from field names and types. A bare JSON Schema carries no comparison configuration at all, so
path B falls back to defaults. That is the price of not writing the model: it still works, but the
thresholds are generic rather than tuned per field.

In [8]:
a_cfg = evaluators["A: hand-written"].explain()
b_cfg = evaluators["B: from dataset"].explain()

print(f"{'field':24} {'A comparator':24} {'A thr':>6}  {'B thr':>6}")
print("-" * 66)
for field in sorted(set(a_cfg) & set(b_cfg)):
    a, b = a_cfg[field], b_cfg[field]
    print(f"{field:24} {a['comparator']:24} {a['threshold']:>6}  {b['threshold']:>6}")

print(f"\nA thresholds: {sorted({c['threshold'] for c in a_cfg.values()})}")
print(f"B thresholds: {sorted({c['threshold'] for c in b_cfg.values()})}")
print("\nPath B is uniform because a JSON Schema says nothing about how to compare a field.")
print("To tune it, export with to_stickler_config(), edit, and reload with model_from_json().")

# explain() misreports List[StructuredModel] fields on an already-configured
# StructuredModel: it names LevenshteinComparator where the engine actually does
# Hungarian per-element matching, and omits the nested paths. The comparison
# itself is correct -- metrics() above shows the nested LineItems.* rows.

field                    A comparator              A thr   B thr
------------------------------------------------------------------
Advertiser               LevenshteinComparator       0.7     0.5
Agency                   LevenshteinComparator       0.7     0.5
AgencyCommission         NumericComparator          0.95     0.5
GrossTotal               NumericComparator          0.95     0.5
LineItems                Hungarian (per-element StructuredModel)    0.7     0.5
NetAmountDue             NumericComparator          0.95     0.5
PaymentTerms             LevenshteinComparator       0.7     0.5

A thresholds: [0.6, 0.7, 0.95]
B thresholds: [0.5]

Path B is uniform because a JSON Schema says nothing about how to compare a field.
To tune it, export with to_stickler_config(), edit, and reload with model_from_json().
